# 🛍️ Decoding Customer Value: Feature Engineering Pipeline
**Lead Data Strategist:** Krrish Kumar  
**Evaluation:** IIT Guwahati Consulting & Analytics Club

---

### 🏛️ Executive Summary
The raw D2C transaction dataset lacks explicit loyalty labels, churn indicators, and chronological timestamps. This pipeline serves as a deterministic feature engineering engine, synthesizing latent behavioral signals (Promo Dependency, Value Tiers, and Loyalty Constructs) into a clean dimensional schema for downstream SQL analytics and Power BI visualization.

### ⚙️ Core Synthesized Features
1. **Promo Dependency Score:** A continuous ratio measuring reliance on discounts.
2. **Value Tiering:** Decile-based categorization of total historical spend.
3. **Loyalty Construct:** A synthesized dual-condition flag requiring both high-frequency purchasing and low-discount reliance.

In [4]:
# ================================================================
# 01. INGESTION & DETERMINISTIC FEATURE SYNTHESIS
# ================================================================
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

print("🚀 Initializing D2C Feature Engineering Pipeline (Local Mode)...")

# Local Relative Path
input_path = 'd2c_fashion_data/transactions.csv'
output_path = 'd2c_fashion_data/clean_customer_dimensions.csv'

try:
    df = pd.read_csv(input_path)
    print(f"✅ Raw D2C Transactions Loaded: {len(df):,} records.")
except FileNotFoundError:
    print(f"❌ CRITICAL ERROR: Ensure '{input_path}' exists.")
    raise

# Standardize column names for SQL compatibility (lowercase, underscores)
df.columns = df.columns.str.lower().str.replace(' ', '_').str.replace('(', '').str.replace(')', '')

# 1. Synthesize Base Customer Metrics
# Note: Kaggle's 'Customer Shopping Trends' uses slightly different column names.
# We map them dynamically here.
customer_metrics = df.groupby('customer_id').agg(
    total_spend=('purchase_amount_usd', 'sum'),
    avg_order_value=('purchase_amount_usd', 'mean'),
    total_transactions=('previous_purchases', 'max'), # Using previous purchases as frequency proxy
    avg_review_score=('review_rating', 'mean')
).reset_index()

# 2. Synthesize Promo Dependency
# Calculating how often a discount or promo code was applied
df['discount_used_flag'] = np.where((df['discount_applied'] == 'Yes') | (df['promo_code_used'] == 'Yes'), 1, 0)
promo_stats = df.groupby('customer_id')['discount_used_flag'].mean().reset_index()
promo_stats.rename(columns={'discount_used_flag': 'promo_dependency_score'}, inplace=True)

customer_metrics = pd.merge(customer_metrics, promo_stats, on='customer_id')

# 3. Synthesize the "Loyalty Construct" (Solving the Prompt's Core Constraint)
# Definition: Top 30% of spenders who rely on promos LESS than 40% of the time.
spend_70th_pct = customer_metrics['total_spend'].quantile(0.70)

conditions = [
    (customer_metrics['total_spend'] >= spend_70th_pct) & (customer_metrics['promo_dependency_score'] <= 0.40),
    (customer_metrics['promo_dependency_score'] >= 0.75),
    (customer_metrics['total_spend'] < spend_70th_pct) & (customer_metrics['promo_dependency_score'] < 0.75)
]
choices = ['True Loyalists', 'Bargain Hunters', 'Standard Customers']
customer_metrics['loyalty_segment'] = np.select(conditions, choices, default='Unclassified')

# 4. Synthesize "Satisfaction Flag"
customer_metrics['satisfaction_flag'] = np.where(customer_metrics['avg_review_score'] >= 4.0, 'High', 'At-Risk')

# Export to clean CSV for SQL Engine
customer_metrics.to_csv(output_path, index=False)
print(f"📊 Feature Engineering Complete. Dimension table exported to: '{output_path}'")
print(f"⚠️ Segment Breakdown:\n{customer_metrics['loyalty_segment'].value_counts(normalize=True)*100}")

🚀 Initializing D2C Feature Engineering Pipeline (Local Mode)...
✅ Raw D2C Transactions Loaded: 3,900 records.
📊 Feature Engineering Complete. Dimension table exported to: 'd2c_fashion_data/clean_customer_dimensions.csv'
⚠️ Segment Breakdown:
loyalty_segment
Bargain Hunters       43.000000
Standard Customers    38.974359
True Loyalists        18.025641
Name: proportion, dtype: float64


In [5]:
# ================================================================
# 01. INGESTION & DETERMINISTIC FEATURE SYNTHESIS
# ================================================================
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

print("🚀 Initializing D2C Feature Engineering Pipeline (Local Mode)...")

# Local Relative Path
input_path = 'd2c_fashion_data/transactions.csv'
output_path = 'd2c_fashion_data/clean_customer_dimensions.csv'

try:
    df = pd.read_csv(input_path)
    print(f"✅ Raw D2C Transactions Loaded: {len(df):,} records.")
except FileNotFoundError:
    print(f"❌ CRITICAL ERROR: Ensure '{input_path}' exists.")
    raise

# Standardize column names for SQL compatibility (lowercase, underscores)
df.columns = df.columns.str.lower().str.replace(' ', '_').str.replace('(', '').str.replace(')', '')

# 1. Synthesize Base Customer Metrics
# We map the Kaggle dataset columns dynamically here.
customer_metrics = df.groupby('customer_id').agg(
    total_spend=('purchase_amount_usd', 'sum'),
    avg_order_value=('purchase_amount_usd', 'mean'),
    total_transactions=('previous_purchases', 'max'), # Using previous purchases as frequency proxy
    avg_review_score=('review_rating', 'mean')
).reset_index()

# 2. Synthesize Promo Dependency
# Calculating how often a discount or promo code was applied
df['discount_used_flag'] = np.where((df['discount_applied'] == 'Yes') | (df['promo_code_used'] == 'Yes'), 1, 0)
promo_stats = df.groupby('customer_id')['discount_used_flag'].mean().reset_index()
promo_stats.rename(columns={'discount_used_flag': 'promo_dependency_score'}, inplace=True)

customer_metrics = pd.merge(customer_metrics, promo_stats, on='customer_id')

# 3. Synthesize the "Loyalty Construct" (Solving the Prompt's Core Constraint)
# Definition: Top 30% of spenders who rely on promos LESS than 40% of the time.
spend_70th_pct = customer_metrics['total_spend'].quantile(0.70)

conditions = [
    (customer_metrics['total_spend'] >= spend_70th_pct) & (customer_metrics['promo_dependency_score'] <= 0.40),
    (customer_metrics['promo_dependency_score'] >= 0.75),
    (customer_metrics['total_spend'] < spend_70th_pct) & (customer_metrics['promo_dependency_score'] < 0.75)
]
choices = ['True Loyalists', 'Bargain Hunters', 'Standard Customers']
customer_metrics['loyalty_segment'] = np.select(conditions, choices, default='Unclassified')

# 4. Synthesize "Satisfaction Flag"
customer_metrics['satisfaction_flag'] = np.where(customer_metrics['avg_review_score'] >= 4.0, 'High', 'At-Risk')

# Export to clean CSV for SQL Engine
customer_metrics.to_csv(output_path, index=False)
print(f"📊 Feature Engineering Complete. Dimension table exported to: '{output_path}'")
print(f"⚠️ Segment Breakdown:\n{customer_metrics['loyalty_segment'].value_counts(normalize=True)*100}")

🚀 Initializing D2C Feature Engineering Pipeline (Local Mode)...
✅ Raw D2C Transactions Loaded: 3,900 records.
📊 Feature Engineering Complete. Dimension table exported to: 'd2c_fashion_data/clean_customer_dimensions.csv'
⚠️ Segment Breakdown:
loyalty_segment
Bargain Hunters       43.000000
Standard Customers    38.974359
True Loyalists        18.025641
Name: proportion, dtype: float64
